# Session 1.2: AI Assistant in Jupyter

JupyterLab / Jupyter Notebooks offers several options for integrating and using AI assistants that can help us with programming and data analysis tasks.

For our training sessions, we want to use the AI assistant [bia-bob](https://github.com/haesleinhuepf/bia-bob), which is provided via a [Python package](https://pypi.org/project/bia-bob/).

In this notebook, we will look at the general use of the AI assistant - from importing and initializing to assisting with difficult tasks.

## API Access

As explained in the slides, we want to use an LLM provider API that our AI assistant can connect to. For this, we need to provide the URL for the LLM endpoint (API), and an according API key. Both will be provided as so-called `environment variables` / `env vars` (a constant key-value pair available in memory).

To check what env vars are set and available right now, we can use the following commands in a code cell:
* `!printenv` (bash/zsh terminal on Linux OS)
* `!set` or `!ls env:` (command prompt or PowerShell on Windows OS)
* `%env` (IPython Magic Command for Notebooks, OS agnostic)

You can also check for a specific env var, e.g. `ENDPOINT_URL`:
* `%env ENDPOINT_URL`

In [ ]:
# ToDo: Check the available env vars on your system

We now want to provide the URL for the LLM endpoint API and the API key as env vars.

To do this for JuypterLab in a virtual environment managed by `uv`:
* Create a file `secrets` in the environments' directory
* Add the line: `ENDPOINT_URL=url` (replace _url_ with the actual URL to the API)
* Add the line: `OPENAI_API_KEY=your-api-key` (replace _your-api-key_ with the actual key)
* Stop JupyterLab
* Restart JupyterLab with the content of this `secrets` file via:
```bash
uv run --env-file path/to/secrets jupyter lab
```

In [ ]:
# Running this cell should now print the endpoint URL
%env ENDPOINT_URL

In [ ]:
# Running this cell should now print your API key
%env OPENAI_API_KEY

## Import and Initialize the AI Assistant

In [1]:
# Import os to read env vars
import os
# Import bia-bob as bob and additional helper functions
from bia_bob import bob, available_models, fix, doc

To intialize bob with any LLM provider, we can set the according URL to its API via the parameter `endpoint`:

In [ ]:
bob.initialize(endpoint=os.getenv('ENDPOINT_URL'))

We can now print all available LLMs for this `endpoint`:

In [4]:
models = available_models(endpoint=os.getenv('ENDPOINT_URL'))
# Show some of the available models
display(models[10:20])

['glm-5.1',
 'gpt-oss-120b',
 'kimi-k2.5',
 'kimi-k2.6',
 'llama-4-scout-17b-16e-instruct',
 'mini',
 'mistral-medium-3.5',
 'multilingual-e5-large-instruct',
 'mxbai-embed-large:latest',
 'nomic-embed-text-v1.5']

We can now initialize bob to use this endpoint and a specific LLM:

In [ ]:
bob.initialize(
    endpoint=os.getenv('ENDPOINT_URL'),
    model="mini",
)

## Use the AI Assistant

### Prompting

The AI assistant can then be used in a code cell via IPython's so-called Magic Commands (`%`, `%%`). 

Either via:

```bash
%bob Single-line prompt. All is written into one line
```

...or via:

```bash
%%bob
Multi-line prompt.
With more details.
And more context.
```


In [54]:
%%bob 
Hello. 
Are you familiar with Machine Learning ? 
Please explain briefly. 
Just a short text output, no code nor code cell !

Yes—I’m familiar with Machine Learning (ML). It is a sub‑field of artificial intelligence that enables computers to learn patterns and make predictions or decisions from data without being explicitly programmed for each task. Typical ML workflows involve (1) gathering and preparing a dataset, (2) selecting a model‑type (e.g., linear regression, decision trees, neural networks), (3) training the model by optimizing its parameters on the data, (4) evaluating performance on held‑out data, and (5) deploying the model to automate or augment real‑world processes. ML can be supervised (learning from labeled examples), unsupervised (discovering structure in unlabeled data), or reinforcement‑based (learning through trial‑and‑error interactions). Its success hinges on high‑quality data, appropriate feature engineering, and careful validation to avoid overfitting or bias.

---
### Prompt augmentation

To inject the content of existing variables into prompts, we can augment a prompt by using `{variable_name}`.

Here we need to consider the size of the variable content, since LLMs only have a limited context window (number of tokens in a prompt and conversation)


In [55]:
%%bob
Read the names of LLMs provided in this list {models}.
For four of these models of your choice, explain what each model is capable of and when it should be used.

**1. deepseek‑v4‑pro (and deepseek‑v4‑pro‑thinking)**  
*Capability*: A 2024‑era large language model (≈130 B parameters) optimized for high‑quality instruction following, reasoning, and code generation. The “‑thinking” variant adds a chain‑of‑thought prompting head that produces intermediate reasoning steps, improving performance on complex logical or mathematical problems.  
*When to use it*:  
- Research‑grade text generation where accuracy and depth matter (e.g., literature reviews, drafting scientific explanations).  
- Tasks that require multi‑step reasoning such as problem‑solving, data‑analysis planning, or explaining code.  
- Projects that need a strong balance of general knowledge and coding ability, and where you have enough GPU memory or can run it via a hosted API.

**2. gemma4**  
*Capability*: Gemma‑4 is a 2024 open‑weight model from Google DeepMind (≈9 B parameters) focused on instruction‑following with an emphasis on safety and factuality. It excels at conversational assistance, summarization, and short‑form content creation, and it is designed to run efficiently on consumer‑grade GPUs.  
*When to use it*:  
- Interactive chatbots, help‑desk assistants, or internal knowledge‑base Q&A where response latency and resource cost are constraints.  
- Situations that need a “compact but safe” model—e.g., deploying on‑premise for privacy‑sensitive internal tools.  
- Rapid prototyping of text‑generation features when you cannot afford the inference cost of >50 B‑parameter models.

**3. qwen3.5 (including qwen3.5‑122b)**  
*Capability*: Part of Alibaba’s Qwen series, Qwen‑3.5 is a multilingual, instruction‑tuned family ranging from 7 B to 122 B parameters. It supports 100+ languages, strong coding abilities (especially for Chinese codebases), and integrated tool‑use (e.g., function calling). The 122 B variant offers state‑of‑the‑art performance on benchmark suites comparable to GPT‑4‑Turbo.  
*When to use it*:  
- Global products that must understand and generate text in many languages, especially when Chinese support is critical.  
- Heavy‑duty applications such as automated report generation, large‑scale content translation, or code assistance for multilingual development teams.  
- Scenarios where you can host the model on high‑end GPU clusters and need top‑tier quality without relying on external commercial APIs.

**4. whisper‑large‑v3**  
*Capability*: Whisper‑large‑v3 is OpenAI’s latest open‑source speech‑to‑text model (≈1.5 B parameters) trained on a massive multilingual audio‑transcription dataset. It delivers near‑human accuracy across 99 languages, handles background noise, speaker diarization, and can produce timestamps and word‑level confidence scores.  
*When to use it*:  
- Converting recordings of interviews, lectures, or meetings into searchable transcripts for research data management.  
- Adding subtitles or closed captions to multimedia assets while complying with accessibility regulations.  
- Building voice‑controlled interfaces or performing audio analytics (e.g., extracting keywords from podcasts) when you need an on‑premise solution that respects data privacy.

---

### Code generation and explanation

However, another useful application of the AI assistant is in generating and explaning code, e.g. for data analysis or exploration tasks. As a general rule, keep in mind that the more specific your prompt is, the better the generated results will be.

You can ask the AI assistant to explain code
* Add the magic command for a multi-line prompt on top of the affected code cell, e.g.: `%%bob Explain the following code in detail for beginners`

You can also ask the AI assistant to add proper documentation to the code
* Import the according function from bia-bob: `from bia_bob import doc`
* Add the magic command `%%doc` on top of the according code cell

**Let's try it out**

Our assistant will use libraries which are not available in your venv yet, .e.g. `pandas`. This will result in an error. You can use `uv` to install (add) it:

In [1]:
#!uv add pandas

In [ ]:
%%bob
Generate Python code to do the following:
- Create a list of 10 random floats
- Compute the mean and the standard deviation
- Print the results
- Make use of the packages numpy and pandas, if necessary
- Just the code, NO comments at all

In [ ]:
import numpy as np
import pandas as pd
rand_floats = np.random.rand(10).tolist()
series = pd.Series(rand_floats)
mean_new = series.mean()
std_new = series.std()

print(rand_floats)
print(mean_new)
print(std_new)

### Fixing code errors

The AI assistant can also be used to `fix` erroneous code:
* Import the according function from bia-bob: `from bia_bob import fix`
* Add the magic command `%%fix` on top of the affected code cell

In [60]:
import numpy as np
import pandas as pd

rand_floats = np.random.rand(10).tolist()
series = pd.Series(rand_flo)
mean_new = series.mean()
std_new = series.std()

print(rand_floats)
print(mean_new)
print(std_new)

NameError: name 'rand_flo' is not defined